## E Commerce Funnel Analysis 
Exploratory Data Analysis
Understanding event distributions, user behaviour 
and patterns before building the funnel.

## Import Libraries  

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

## Load Data 

In [2]:
df = pd.read_csv('data/ecommerce_cleaned.csv')
print(f"Dataset loaded")
print(f"Shape: {df.shape}")
df.head()

Dataset loaded
Shape: (884964, 13)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,date,hour,month_name,day_of_week
0,2020-09-24 11:57:06+00:00,view,1996170,2144415922528452715,electronics.telephone,Uncategorised,31.90,1515915625519388267,LJuJVLEjPT,2020-09-24,11,September,Thursday
1,2020-09-24 11:57:26+00:00,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY,2020-09-24,11,September,Thursday
2,2020-09-24 11:57:27+00:00,view,215454,2144415927158964449,Uncategorised,Uncategorised,9.81,1515915625513238515,4TMArHtXQy,2020-09-24,11,September,Thursday
3,2020-09-24 11:57:33+00:00,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08,2020-09-24,11,September,Thursday
4,2020-09-24 11:57:36+00:00,view,3658723,2144415921169498184,Uncategorised,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ,2020-09-24,11,September,Thursday


## Exploratory Data Analysis 
In this section we explore the cleaned dataset through 
visualisations and statistical summaries to understand 
user behaviour and identify patterns before building 
the funnel.

## 1. Event Type Distribution 
Analysing the breakdown of event types to understand 
the overall funnel shape

###### Count of each event type 

In [3]:
event_counts = df['event_type'].value_counts().reset_index()
event_counts.columns = ['event_type', 'count']
print(event_counts)

  event_type   count
0       view  793589
1       cart   54029
2   purchase   37346


In [4]:
fig = px.bar(
    event_counts,
    x='event_type',
    y='count',
    title='EVENT TYPE DISTRIBUTION',
    color='event_type',
    color_discrete_map={
   'view': 'steelblue',
   'cart': 'skyblue',
   'purchase': 'salmon'
    },
    labels={'count': 'Number of Events',
    'event_type': 'Event Type'}
)
fig.update_layout()
fig.show()

##  Key Insights:

- View events account for the vast majority of activity at 
793,589 (89.7%), followed by cart events at 54,029 (6.1%) 
and purchase events at 37,346 (4.2%).

- For every 100 users who view a product, only 6 add to cart 
and only 4 complete a purchase, confirming a clear funnel 
drop-off pattern.
## Business Implication
The biggest conversion opportunity lies in the view-to-cart 
stage where 91% of users drop off. Improving product pages, 
adding urgency signals and implementing personalised 
recommendations could significantly improve the current 
6% cart rate and recover lost revenue.

## 2. Event Distribution by Hour
Analysing which hours of the day see the most activity

In [5]:
hourly_events = df.groupby(
    ['hour', 'event_type']
).size().reset_index(name='count')
hourly_events


,hour,event_type,count
0,0,cart,584
1,0,purchase,361
2,0,view,9346
3,1,cart,532
4,1,purchase,328
...,...,...,...
67,22,purchase,701
68,22,view,15431
69,23,cart,694
70,23,purchase,482


In [6]:
fig = px.line(
    hourly_events,
    x='hour',
    y='count',
    color='event_type',
    title='EVENT DISTRIBUTION BY HOUR OF DAY',
    labels={'count': 'Number of Events',
   'hour': 'Hour of Day (0=Midnight, 12=Noon)'},
    color_discrete_map={
    'view': 'steelblue',
  'cart': 'salmon',
   'purchase': 'skyblue'
    },
    markers=True
)
fig.update_xaxes(tickvals=list(range(0, 22)))
fig.show()

## Key Insights:
- View events rise sharply from 7am, peak between 10am and 6pm maintaining consistently high traffic of 40k-45k events
- Activity drops steadily after 6pm (hour 18) with no secondary evening peak
- Cart and purchase events remain significantly lower than views throughout the entire day, indicating a conversion gap at all hours
- Lowest activity occurs between 1am and 5am across all event types
- Peak shopping window is daytime hours (10am-6pm)
## Business Implication
The clear daytime peak between 10am and 6pm represents the 
highest opportunity window for conversion-focused campaigns. 
Scheduling promotional emails, flash sales and targeted ads 
during these hours would reach customers at their most active 
and engaged. The sharp drop after 6pm suggests limited value 
in evening marketing spend, while the 1am to 5am window should 
be used for backend operations such as inventory updates and 
system maintenance rather than customer-facing activity.

## Top 10 Category by Events 
Analysing which product categories drive the most activity.

In [7]:
top_categories = df[
    df['category_code'] != 'Uncategorised'
].groupby('category_code').size().reset_index(
    name='count'
).sort_values('count', ascending=False).head(10).sort_values('count', ascending=True)
top_categories

,category_code,count
87,electronics.tablet,19376
90,electronics.video.tv,21391
39,computers.components.cpu,24768
51,computers.notebook,25025
41,computers.components.motherboard,26600
77,electronics.audio.acoustic,26764
104,stationery.cartrige,38719
58,computers.peripherals.printer,43219
88,electronics.telephone,84343
46,computers.components.videocards,116712


In [8]:
fig = px.bar(
    top_categories,
    x='count',
    y='category_code',
    orientation='h',
    title='TOP 10 PRODUCT CATEGORIES BY EVENTS',
    labels={'count': 'Number of Events',
            'category_code': 'Category'},
    color_discrete_sequence=['steelblue']
)
fig.show()

## Key Insights:
- Computers.components.videocards dominates platform activity with 116,712 events, nearly double electronics.telephone at 84,343, confirming strong GPU demand. 
- The top 10 categories being exclusively electronics and computer components confirms this is a specialist technical store rather than a generalretailer. 
- Printer and cartridge categories appearing together suggests a loyal recurring customer base buying consumables alongside hardware.

## Business Implication
Marketing budget should prioritise GPU components and 
smartphones as primary revenue drivers. The recurring 
printer and cartridge purchases present an opportunity 
for a subscription model to lock in repeat customers.

## Brand Performance 
Analysing which brands generate the most events 
and comparing views vs purchases by brand

In [9]:
top_brands = df[
    df['brand'] != 'Uncategorised'
].groupby('brand').size().reset_index(
    name='count'
).sort_values('count', ascending=False).head(10)

print(top_brands)

         brand  count
60        asus  27703
318   gigabyte  27673
591        msi  24876
664      palit  24801
766    samsung  23198
34         amd  20107
136      canon  18437
666  panasonic  11986
686    pioneer  11467
796     sirius  11404


In [10]:
top_brands_sorted = top_brands.sort_values('count', ascending=True)

fig = px.bar(
    top_brands,
    x='brand',
    y='count',
    title='TOP 10 BRANDS BY EVENT COUNT',
    labels={'count': 'Number of Events', 'brand': 'Brand'},
    color_discrete_sequence=['steelblue']
)
fig.show()

## Key Insights:
- Top brands are dominated by computer components 
  manufacturers. ASUS, Gigabyte, MSI, Palit
- These are GPU and motherboard brands suggesting 
  this store's core customer is a PC builder
- Samsung is the only mainstream consumer brand 
  in the top 10, indicating niche technical audience
- Canon and Panasonic presence suggests some 
  photography and audio segment exists

## Business implication:
Marketing should target PC enthusiasts and tech builders rather than general consumers

## Price Sensitivity Analysis 
Comparing average prices across event types to 
understand if price influences conversion.

In [11]:
price_analysis = df.groupby(
    'event_type'
)['price'].mean().reset_index()
price_analysis.columns = ['event_type', 'avg_price']

print(price_analysis)

  event_type   avg_price
0       cart  159.647668
1   purchase  137.240819
2       view  145.840013


In [12]:
fig = px.bar(
    price_analysis,
    x='event_type',
    y='avg_price',
    title='AVERAGE PRICE BY EVENT TYPE',
    labels={'avg_price': 'Average Price (£)',
     'event_type': 'Event Type'},
    color='event_type',
    color_discrete_map={
  'view': 'salmon',
   'cart': 'steelblue',
   'purchase': 'skyblue'
    }
)
fig.show()

## Key Insights: 
- Cart items carry the highest average price at £159.65, followed by viewed items at £145.84, with purchased items 
having the lowest average at £137.24. 
- The £22.41 gap between cart and purchase price confirms that customers add premium items to cart but ultimately settle for lower priced alternatives at checkout, revealing a clear price sensitivity threshold in the final purchase decision.

## Business Implication
Offering targeted discounts or Buy Now Pay Later options on 
abandoned cart items above £150 could bridge this £22.41 
price gap and recover a significant portion of lost revenue 
at the checkout stage.